# 第10周 b｜术语、单位与队列重建

学习目标：词典映射、未知值审查、单位换算、缺失和信息丢失。

> **本周怎么学**：先学精讲 `w10a_emr_cleaning.ipynb`，再做这份**工程实践**。

__数据：默认使用官方 PRIME-CVD 5 万人数据__（`python run.py download` 下载，来源与 MD5 核验记录在 `data/raw/manifest.json`）。把 `MODE` 改成 `"demo"` 可以切换到离线演示数据，但演示数据是另外生成的，**不是 PRIME-CVD**，结果不能当作官方结果。

本周验收：提交清洗规则和重建误差表；明确BMI/精确随访是否可恢复。

In [1]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
from IPython.display import display

# 从项目根目录、notebooks目录或其他项目内目录启动均可。
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "run.py").exists()), None)
if ROOT is None:
    raise RuntimeError("请在解压后的项目目录中启动 Jupyter。")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from primecvd.common import load_config, provenance_label
from primecvd.pipeline import prepare
from primecvd.splitting import select_split

MODE = "official"  # 官方数据；离线练习或跑测试时可改为 "demo"（演示数据不是 PRIME-CVD）
cfg = load_config(ROOT / "configs" / f"{MODE}.json")
cfg["output_dir"] = f"outputs/notebooks_{MODE}"
print(provenance_label(cfg))
df, splits = prepare(cfg)
train = select_split(df, splits, "train")
valid = select_split(df, splits, "valid")
# 不创建test DataFrame；模型试验只使用train和valid。
print("队列、训练集、验证集：", len(df), len(train), len(valid))

# 命令行绘图模块采用无窗口后端；在Notebook中显式启用行内图像。
get_ipython().run_line_magic("matplotlib", "inline")
from primecvd import pubplot as pp   # 与精讲篇相同的顶刊作图风格（在启用行内图像之后设置）
pp.setup()

OFFICIAL SOURCE FILES / 官方下载文件
队列、训练集、验证集： 50000 35000 7500


## 清洗不是随便填齐表格
映射疾病/测量名称，检查单位，验证主键。遇到未知术语或不明单位应停下来审查，而不是默认为正常。`Age_At_2024-7`是作者这个版本的构造规则，不是通用年龄处理公式。

In [2]:
from primecvd.data import load_emr
from primecvd.reconstruction import reconstruct, compare_to_clean
rec, audit, standardized_measurements = reconstruct(load_emr(cfg))
display(rec.head())
display(pd.Series(audit))
display(compare_to_clean(rec, df))

,Patient_ID,Age,IRSD_quintile,smoking_status,cvd_event,CVD_Time_month_recorded,diabetes,CKD,AF,BMI,HbA1c,Height,SBP,Weight,eGFR,BMI_source,coarsened_time_midpoint_years_NOT_original
0,269,50.40,4,non,1,2019-12,0,0,0,28.194773,4.317890,158.8,118.668194,71.1,83.077560,derived_from_height_weight,2.958333
1,272,39.23,3,ex,0,2022-12,0,0,0,16.818800,4.700951,167.7,123.719732,47.3,81.488401,derived_from_height_weight,5.958333
2,281,55.49,5,non,0,2022-12,0,0,0,23.520000,3.669685,NaN,126.650517,NaN,86.779230,recorded,5.958333
3,296,51.91,4,non,0,2022-12,0,0,0,31.980000,4.486977,NaN,113.889670,NaN,94.704093,recorded,5.958333
4,317,47.09,1,ex,0,2022-12,0,0,0,25.350000,4.315440,NaN,125.912615,NaN,86.336256,recorded,5.958333


hba1c_mmol_mol_rows                                                                             2500
unit_conversion_enabled                                                                         True
height_feet_inch_rows                                                                           1000
bmi_rows_without_unit                                                                          40000
description_unmapped_but_canonical_measure_used                                                    0
master_rows                                                                                    50000
disease_rows                                                                                    4417
measurement_rows                                                                              210000
reconstructed_rows                                                                             50000
unknown_smoking                                                                            

,variable,n_comparable,n_unavailable,mean_absolute_error,max_absolute_error,tolerance,within_tolerance_fraction
0,Age,50000,0,2.478163e-03,4.999992e-03,0.011,1.00000
1,BMI,50000,0,8.570463e-03,6.300014e-01,0.011,0.89644
2,HbA1c,50000,0,4.263256e-18,1.776357e-15,0.011,1.00000
3,eGFR,50000,0,0.000000e+00,0.000000e+00,0.011,1.00000
4,SBP,50000,0,0.000000e+00,0.000000e+00,0.011,1.00000
5,IRSD_quintile,50000,0,0.000000e+00,0.000000e+00,0.000,1.00000
6,diabetes,50000,0,0.000000e+00,0.000000e+00,0.000,1.00000
7,CKD,50000,0,0.000000e+00,0.000000e+00,0.000,1.00000
8,AF,50000,0,0.000000e+00,0.000000e+00,0.000,1.00000
9,cvd_event,50000,0,0.000000e+00,0.000000e+00,0.000,1.00000


这里使用作者结构规则建立患者映射，并先检查共同字段。在官方数据上，诊断、HbA1c、eGFR、SBP 与 Asset1 完全一致；BMI 中有 20% 由身高体重派生，带有舍入误差。这只说明清洗规则正确，**不是外部验证**：两份数据是同一批合成人员。

In [3]:
print("重建后患者数：", len(rec))
print("BMI 来源：", rec.BMI_source.value_counts().to_dict())
print("BMI缺失比例：", rec.BMI.isna().mean())
print("是否输出了可冒充原始随访的cvd_time：", "cvd_time" in rec.columns)
display(standardized_measurements.head())

重建后患者数： 50000
BMI 来源： {'recorded': 40000, 'derived_from_height_weight': 10000}
BMI缺失比例： 0.0
是否输出了可冒充原始随访的cvd_time： False


,Patient_ID,Value,Description,Date,Unit,canonical_measure,standard_value
0,819127997,78.08730821809296,eGFR,2014-03,mL/min/1.73m²,eGFR,78.087308
1,3847713176,8.369465931621287,HbA1c,2015-02,%,HbA1c,8.369466
2,3508304696,140.32800536506784,SBP,2016-09,mmHg,SBP,140.328005
3,1685212472,69.9920201360966,GFR,2015-06,mL/min/1.73m²,eGFR,69.992020
4,1105805072,29.95,BMI,2015-04,NaN,BMI,29.950000


## 明确不可恢复的信息
官方病历表中 80% 的人直接记录了 BMI，另外 20% 只记录了身高和体重（其中一部分身高用英尺英寸记录）。重建器按定义 BMI = 体重 / 身高² 派生，并在 `BMI_source` 列写明来源；只有身高或只有体重时保留缺失，**不从 Asset1 回填**。病历版的月份日期以及被统一的非事件随访结束月份也不能恢复原始连续随访时间。输出中明确标注的粗化时间只用于观察，不进入主要生存模型（第 10 周 a 的 7.2 节量化了用它代替真实随访会产生多大偏差）。

## 独立练习
检查实际输入表有哪些单位和名称。尝试把一项单位改成未知值（只在内存副本中修改），确认程序报错。写下哪些信息能恢复，哪些只能近似，哪些根本不能恢复。

---
本周复盘：请用自己的话记录一个学会的概念、一个发现的错误和一个待解决问题。

特别提醒：不能从Asset1偷偷回填后再声称全部由EMR重建。